# BBDE 301366 — DQ Check

## Purpose
Data availability checks for all source tables and columns used in BBDE 301366 metric calculations.

## Approach
- Uses the **same `check_column()` / `insertDQTable()` pattern** as TDI and ABAC DQ notebooks
- **Union temp views** are created for the segment population (retail + business + TDS)
  so the DQ check reflects the actual source used by the metrics
- Centralized metrics are checked against their respective centralized tables

## Metrics Summary

### Segment Metrics
| Metric | Description |
|--------|-------------|
| SD1 | Total unique customers |
| CDE 1.2 (segment) | Total customer count |
| SD3 | Non-resident customers |
| 6.5A/6.5B | Period start / end |
| 1.6 | Sanctions country exposure |
| SD4 | Occupation risk |
| SD5 | Legal structure |
| 4.1A | Accounts |
| 5.1, 5.6–5.8 | Branches / sanctions jurisdictions |

### Centralized Metrics
| Metric | Source Table | Result |
|--------|-------------|--------|
| CDE 1.3 (Tier 1/2) | scorable_rated_ca | 20,928 |
| CDE 1.2 (High) | scorable_rated_ca | *(captured)* |
| CDE 1.4 (Medium) | scorable_rated_ca | 97,470 |
| CDE 1.5 (Low+Unrated) | scorable_rated_ca + unrated | ~21,229,014 |
| SD2 (PEP) | pep_list_2025_exploded | 20,786 |
| CDE 1.7 (Sanctions) | true_sanction_match_2025 | 5 |
| CDE 1.8 (Blocked) | customer_sanctioned_blocked_property_2025 | 2 |
| SD6 (LYR) | cde_sd_6_lyr_fy2025 | 67,118 |
| CDE 3.17 (UTR) | TD_UTR_CDE_3_17_2025_Cust_details | 12,434 |
| CDE 3.18 (STR/SAR) | td_sar_cde_3_18_2025 | 5,050 |
| CDE 3.19 (LCTR) | cde_3_19_lctr | 107,741 |

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

---
## Shared Functions

In [ ]:
from datetime import date

TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())

lob_id = '301366'
lob_desc = 'BBDE'

def insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                  lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct, today_date):
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True

def check_column(cde_no, cde_desc, source, src_table, column):
    """Check availability of a column and insert into DQ table."""
    try:
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({column} as STRING) IS NOT NULL
                         AND trim(cast({column} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                      lob_id, cde_no, cde_desc, source, src_table,
                      column, pct, today_date)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")

print(f'DQ Check for {lob_id} ({lob_desc})')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

---
## Step 1: Create Union Temp Views
Union all retail, business, and TDS LOB tables into temp views.
These unions are the actual source for BBDE segment metrics.

In [ ]:
# ============================================================
# Retail Union: EDB + PSI + PL + RESL + CC + TDIS
# All aliased to common schema: cust_id, fulln, birth_dt, acct_id
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_retail_union AS
    -- EDB
    SELECT cast_cust_no AS cust_id,
           upper(concat(first_nm, ' ', last_nm)) AS fulln,
           birth_dt, acct_id, 'EDB' AS lob_source
    FROM RA_FY_2025.EDB_FULL_GBL_BUSINESS
    WHERE cbs_effectv_dt = '2025-10-31'
      AND acct_type_cd = 2235 AND lfstgy_cd = 115
    UNION ALL
    -- PSI
    SELECT cas_cust_id AS cust_id,
           upper(concat(first_nm, ' ', last_nm)) AS fulln,
           birth_dt, acct_id, 'PSI' AS lob_source
    FROM RA_FY_2025.PSI_FULL_GBL_BUSINESS
    WHERE abs_effectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
      AND abs_lfstgy_cd IN (114, 115, 117)
    UNION ALL
    -- PL
    SELECT cast_cust_no AS cust_id,
           upper(concat(first_nm, ' ', last_nm)) AS fulln,
           birth_dt, acct_id, 'PL' AS lob_source
    FROM RA_FY_2025.PL_FULL_GBL
    WHERE cbs_effectv_dt = '2025-10-31'
    UNION ALL
    -- RESL
    SELECT cas_cust_id AS cust_id,
           upper(concat(first_nm, ' ', last_nm)) AS fulln,
           birth_dt, acct_id, 'RESL' AS lob_source
    FROM RA_FY_2025.RESL_FULL_GBL
    WHERE cbs_effectv_dt = '2025-10-31'
      AND lfstgy_cd = 115
    UNION ALL
    -- CC
    SELECT cas_cust_id AS cust_id,
           upper(concat(first_nm, ' ', last_nm)) AS fulln,
           birth_dt, acct_id, 'CC' AS lob_source
    FROM RA_FY_2025.CC_FULL_GBL
    WHERE cbs_effectv_dt = '2025-10-31'
    UNION ALL
    -- TDIS
    SELECT ca_cust_id AS cust_id,
           upper(concat(first_nm, ' ', last_nm)) AS fulln,
           birth_dt, acct_id, 'TDIS' AS lob_source
    FROM RA_FY_2025.TDIS_FULL_GBL_BUSINESS
    WHERE abs_effectv_dt = '2025-10-31'
      AND abs_lfstgy_cd IN (114, 115, 117)
""")
print(f'bbde_retail_union: {spark.sql("SELECT count(1) FROM bbde_retail_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# Business Union: SBB_DP + COM_DP + SBB_Credit + COM_Credit
# All aliased to: cust_id (= cif_business_customer_key)
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_business_union AS
    SELECT cif_business_customer_key AS cust_id,
           right(cif_business_customer_key, 8) AS cif_right8,
           'SBB_DP' AS lob_source
    FROM ra_fy_2025.sbb_dp_final
    UNION ALL
    SELECT cif_business_customer_key AS cust_id,
           right(cif_business_customer_key, 8) AS cif_right8,
           'COM_DP' AS lob_source
    FROM ra_fy_2025.com_dp_final
    UNION ALL
    SELECT cif_business_customer_key AS cust_id,
           right(cif_business_customer_key, 8) AS cif_right8,
           'SBB_CR' AS lob_source
    FROM ra_fy_2025.sbb_credit_2025
    UNION ALL
    SELECT cif_business_customer_key AS cust_id,
           right(cif_business_customer_key, 8) AS cif_right8,
           'COM_CR' AS lob_source
    FROM ra_fy_2025.com_credit_2025
""")
print(f'bbde_business_union: {spark.sql("SELECT count(1) FROM bbde_business_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# TDS Union: TDS_201037 (Global Markets) + TDS_201042 (Trade Finance)
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_tds_union AS
    SELECT ROOT_GT_ID AS cust_id,
           ROOT_LEGAL_NAME, ROOT_OPEN_DATE, GT_ACCT_ID,
           RISK_LEVEL, 'TDS_GM' AS lob_source
    FROM RA_FY25_VIEW.TDS_201037
    UNION ALL
    SELECT ROOT_GT_ID AS cust_id,
           ROOT_LEGAL_NAME, ROOT_OPEN_DATE, GT_ACCT_ID,
           RISK_LEVEL, 'TDS_TF' AS lob_source
    FROM RA_FY25_VIEW.TDS_201042
""")
print(f'bbde_tds_union: {spark.sql("SELECT count(1) FROM bbde_tds_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# Dedup / Common Client tables
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_dedup_union AS
    SELECT cust_id, edw_customer_id, 'BBDE_COMMON' AS source
    FROM ra_adido_2025.bbde_retail1_common_client_final
    UNION ALL
    SELECT REPLACE(Customer_Identifier_ID, '.', '') AS cust_id,
           NULL AS edw_customer_id, 'TDS_COMMON' AS source
    FROM ra_adido_2025.tds_retail_common_clients
""")
print(f'bbde_dedup_union: {spark.sql("SELECT count(1) FROM bbde_dedup_union").collect()[0][0]:,} rows')

---
## Step 2: Segment DQ Checks
Check column availability on the **union temp views** (the actual metric source).

In [ ]:
# ============================================================
# Retail Union — primary columns used by segment metrics
# ============================================================
print('bbde_retail_union')
print('=' * 60)

check_column('SD1,CDE1.2,1.5', 'Segment', 'Union:Retail',
             'bbde_retail_union', 'cust_id')

check_column('SD1,1.7,3.17,3.18', 'Segment', 'Union:Retail',
             'bbde_retail_union', 'fulln')

check_column('3.17,6.5A,6.5B', 'Segment', 'Union:Retail',
             'bbde_retail_union', 'birth_dt')

check_column('4.1A', 'Segment', 'Union:Retail',
             'bbde_retail_union', 'acct_id')

check_column('Debug', 'Segment', 'Union:Retail',
             'bbde_retail_union', 'lob_source')

In [ ]:
# ============================================================
# Business Union — primary columns
# ============================================================
print('bbde_business_union')
print('=' * 60)

check_column('SD1,CDE1.2,1.5', 'Segment', 'Union:Business',
             'bbde_business_union', 'cust_id')

check_column('1.2,1.3,1.4', 'Segment', 'Union:Business',
             'bbde_business_union', 'cif_right8')

In [ ]:
# ============================================================
# TDS Union — primary columns
# ============================================================
print('bbde_tds_union')
print('=' * 60)

check_column('SD1,1.2,1.3,1.4', 'Segment', 'Union:TDS',
             'bbde_tds_union', 'cust_id')

check_column('1.7,3.17,3.18', 'Segment', 'Union:TDS',
             'bbde_tds_union', 'ROOT_LEGAL_NAME')

check_column('6.5A,6.5B', 'Segment', 'Union:TDS',
             'bbde_tds_union', 'ROOT_OPEN_DATE')

check_column('4.1A', 'Segment', 'Union:TDS',
             'bbde_tds_union', 'GT_ACCT_ID')

check_column('1.2,1.3,1.4', 'Segment', 'Union:TDS',
             'bbde_tds_union', 'RISK_LEVEL')

In [ ]:
# ============================================================
# Dedup Union — primary columns
# ============================================================
print('bbde_dedup_union')
print('=' * 60)

check_column('Dedup', 'Segment', 'Union:Dedup',
             'bbde_dedup_union', 'cust_id')

check_column('Dedup', 'Segment', 'Union:Dedup',
             'bbde_dedup_union', 'edw_customer_id')

---
## Step 3: Reference / Lookup Tables

In [ ]:
# ============================================================
# EDW Customer (for dedup convert)
# ============================================================
print('RA_FY_2025.edw_customer')
print('=' * 60)

check_column('Dedup', 'Reference', 'Rahona',
             'RA_FY_2025.edw_customer', 'edw_customer_id')

check_column('Dedup', 'Reference', 'Rahona',
             'RA_FY_2025.edw_customer', 'bor_account_id')

In [ ]:
# ============================================================
# Sanctions country risk rating
# ============================================================
print('ra_adido_2025.sanctions_country_risk_rating_2025')
print('=' * 60)

check_column('1.6,5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'country')

check_column('1.6,5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Reference', 'ADIDO',
             'ra_adido_2025.sanctions_country_risk_rating_2025', 'riskrating')

In [ ]:
# ============================================================
# Branch data
# ============================================================
print('ra_adido_2025.bbde_branches_fy2025')
print('=' * 60)

check_column('5.1,5.9', 'Branch', 'ADIDO',
             'ra_adido_2025.bbde_branches_fy2025', 'branch_code')

check_column('5.2,5.3,5.4,5.5,5.6,5.7,5.8', 'Branch', 'ADIDO',
             'ra_adido_2025.bbde_branches_fy2025', 'country')

---
## Step 4: Centralized Metric Sources
Check columns on the centralized tables used by CDE 1.2–1.5, SD2, 1.7, 1.8, SD6, 3.17–3.19.

In [ ]:
# ============================================================
# CRR: scorable_rated_ca (CDE 1.2, 1.3, 1.4)
# ============================================================
print('rafy2025_centralized.scorable_rated_ca')
print('=' * 60)

check_column('1.2,1.3,1.4,1.5', 'Centralized', 'Rahona',
             'rafy2025_centralized.scorable_rated_ca', 'v_entity_id')

check_column('1.2,1.3,1.4', 'Centralized', 'Rahona',
             'rafy2025_centralized.scorable_rated_ca', 'risk_level')

check_column('1.2,1.3,1.4,1.5', 'Centralized', 'Rahona',
             'rafy2025_centralized.scorable_rated_ca', 'v_cust_type_cd')

In [ ]:
# ============================================================
# SD2 (PEP): pep_list_2025_exploded
# ============================================================
print('ra_adido_2025.pep_list_2025_exploded')
print('=' * 60)

check_column('SD2', 'Centralized', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'entity')

check_column('SD2', 'Centralized', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'first_nm')

check_column('SD2', 'Centralized', 'ADIDO',
             'ra_adido_2025.pep_list_2025_exploded', 'last_nm')

In [ ]:
# ============================================================
# CDE 1.7: true_sanction_match_2025
# ============================================================
print('ra_adido_2025.true_sanction_match_2025')
print('=' * 60)

check_column('1.7', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'Customer')

check_column('1.7', 'Centralized', 'ADIDO',
             'ra_adido_2025.true_sanction_match_2025', 'CustomerType')

In [ ]:
# ============================================================
# CDE 1.8: customer_sanctioned_blocked_property_2025
# ============================================================
print('ra_adido_2025.customer_sanctioned_blocked_property_2025')
print('=' * 60)

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIMPACTED')

check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'ACCOUNTBLOCKED')

In [ ]:
# ============================================================
# SD6 (LYR): cde_sd_6_lyr_fy2025
# ============================================================
print('rafy2025_centralized.cde_sd_6_lyr_fy2025')
print('=' * 60)

check_column('SD6', 'Centralized', 'Rahona',
             'rafy2025_centralized.cde_sd_6_lyr_fy2025', 'v_entity_id')

In [ ]:
# ============================================================
# CDE 3.17 (UTR): TD_UTR_CDE_3_17_2025_Cust_details
# ============================================================
print('rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details')
print('=' * 60)

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'full_name')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_type_mn')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt')

check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'account_type')

In [ ]:
# ============================================================
# CDE 3.18 (STR/SAR): td_sar_cde_3_18_2025
# ============================================================
print('rafy2025_centralized.td_sar_cde_3_18_2025')
print('=' * 60)

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'STR_Admin_Matched_CustomerID')

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'STR_Admin_Matched_Customer_Type')

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'STR_LETUA_Customer_Account_Number')

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'customer_first_name')

check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'customer_last_name')

In [ ]:
# ============================================================
# CDE 3.19 (LCTR): cde_3_19_lctr
# ============================================================
print('rafy2025_centralized.cde_3_19_lctr')
print('=' * 60)

check_column('3.19', 'Centralized', 'Rahona',
             'rafy2025_centralized.cde_3_19_lctr', 'account_number')

check_column('3.19', 'Centralized', 'Rahona',
             'rafy2025_centralized.cde_3_19_lctr', 'client_name')

---
## Step 5: Row Count Summary

In [ ]:
# ============================================================
# Row counts for all source tables
# ============================================================
tables = [
    # Retail LOBs (individual)
    ('RA_FY_2025.EDB_FULL_GBL_BUSINESS', 'Retail: EDB'),
    ('RA_FY_2025.PSI_FULL_GBL_BUSINESS', 'Retail: PSI'),
    ('RA_FY_2025.PL_FULL_GBL', 'Retail: PL'),
    ('RA_FY_2025.RESL_FULL_GBL', 'Retail: RESL'),
    ('RA_FY_2025.CC_FULL_GBL', 'Retail: CC'),
    ('RA_FY_2025.TDIS_FULL_GBL_BUSINESS', 'Retail: TDIS'),
    # Business LOBs
    ('ra_fy_2025.sbb_dp_final', 'Business: SBB DP'),
    ('ra_fy_2025.com_dp_final', 'Business: COM DP'),
    ('ra_fy_2025.sbb_credit_2025', 'Business: SBB Credit'),
    ('ra_fy_2025.com_credit_2025', 'Business: COM Credit'),
    # TDS
    ('RA_FY25_VIEW.TDS_201037', 'TDS: Global Markets'),
    ('RA_FY25_VIEW.TDS_201042', 'TDS: Trade Finance'),
    # Dedup
    ('ra_adido_2025.bbde_retail1_common_client_final', 'Dedup: BBDE Common'),
    ('ra_adido_2025.tds_retail_common_clients', 'Dedup: TDS Common'),
    ('RA_FY_2025.edw_customer', 'Dedup: EDW'),
    # Centralized
    ('rafy2025_centralized.scorable_rated_ca', 'CRR: Rated'),
    ('ra_adido_2025.pep_list_2025_exploded', 'SD2: PEP'),
    ('ra_adido_2025.true_sanction_match_2025', 'CDE 1.7: Sanctions'),
    ('ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CDE 1.8: Blocked'),
    ('rafy2025_centralized.cde_sd_6_lyr_fy2025', 'SD6: LYR'),
    ('rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'CDE 3.17: UTR'),
    ('rafy2025_centralized.td_sar_cde_3_18_2025', 'CDE 3.18: STR/SAR'),
    ('rafy2025_centralized.cde_3_19_lctr', 'CDE 3.19: LCTR'),
    # Branches
    ('ra_adido_2025.bbde_branches_fy2025', 'Branches: BBDE'),
    ('ra_adido_2025.tds_branches_location_FY25', 'Branches: TDS'),
    # Reference
    ('ra_adido_2025.sanctions_country_risk_rating_2025', 'Reference: Sanctions Country'),
    # Union temp views
    ('bbde_retail_union', 'UNION: Retail'),
    ('bbde_business_union', 'UNION: Business'),
    ('bbde_tds_union', 'UNION: TDS'),
    ('bbde_dedup_union', 'UNION: Dedup'),
]

print(f'{"Table":<65} {"Description":<30} {"Row Count":>15}')
print('=' * 115)
for table, desc in tables:
    try:
        cnt = spark.sql(f'SELECT count(1) FROM {table}').collect()[0][0]
        print(f'{table:<65} {desc:<30} {cnt:>15,}')
    except Exception as e:
        print(f'{table:<65} {desc:<30} {"ERROR":>15}')

---
## Step 6: Summary

In [ ]:
# ============================================================
# Display all DQ results for this AU
# ============================================================
results = spark.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE lob_id = '{lob_id}'
      AND run_date = '{today_date}'
    ORDER BY src_table_name, data_element
""")

print(f'Total DQ checks: {results.count()}')
display(results)

---
## N/A Metrics

| Metric | Reason |
|--------|--------|
| 6.1, 6.2, 6.3, 6.4, 6.6, 6.6B | Not calculated in any of CPB, TDI, TDW |
| 5.9 | All branches counted (no high crime rate area list) |

## TDW / TDS Applicability

| Metric | TDW DI/PB | TDS TF |
|--------|-----------|--------|
| CDE 3.17 (UTR) | NOT applicable | N/A |
| CDE 3.18 (STR/SAR) | PB NOT applicable | N/A |
| CDE 3.19 (LCTR) | NOT applicable | No TDS TF queries |

In [ ]:
# ============================================================
# Cleanup temp views
# ============================================================
for view in ['bbde_retail_union', 'bbde_business_union', 'bbde_tds_union', 'bbde_dedup_union']:
    spark.sql(f'DROP VIEW IF EXISTS {view}')
    print(f'Dropped: {view}')
print('Cleanup complete.')